In [1]:
from pathlib import Path
import os
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))
data_path = PROJECT_ROOT / 'results' / 'input_distribution'

from utils import olc_client
c = olc_client.connect(verbose=True)

Connected to https://neuprint-cns.janelia.org[cns].
Client: neuprint-python v0.1.0
User: hoejud@gmail.com [readwrite]



In [2]:
import numpy as np
import plotly.graph_objects as go

from neuprint import NeuronCriteria as NC, SynapseCriteria as SC, fetch_synapses, fetch_synapse_connections, fetch_neurons
from utils.hex_geometry import find_depth, find_hex_ids, find_neuron_hex_ids, load_depth_bins
from utils.query_functions import fetch_top_input_instances
from utils.input_distr_functions import find_1d_input_distrs, fetch_n_find_rel_hex, find_rel_hex
from utils.plotting_functions import plot_1d_polar, plot_1d_distr, plot_col_heatmap

In [19]:
instance_name = 'T4c'+'_R'
neuropil = 'ME(R)'
hex0=45

### home column for post-synapses

In [ ]:
syn_df = fetch_synapses(NC(instance=instance_name), SC(rois=[neuropil], type='post'))

hex_df = find_neuron_hex_ids(syn_df, roi_str=neuropil, method='majority')#, method='majority') #, method='COM')
hex_red_df = hex_df[['bodyId','hex1_id','hex2_id']]\
    .rename(columns={'bodyId': 'bodyId_post'})

In [5]:
n_out = hex_red_df['bodyId_post'].nunique()
print('Number of %s neurons: %d'%(instance_name, n_out))

  0%|          | 0/85 [00:00<?, ?it/s]

Number of T4a_R neurons: 849


### distribution of all input connections

In [6]:
conn_control_df = fetch_synapse_connections(None, \
                                    NC(instance=instance_name) ,\
                                    SC(rois=[neuropil]))
syn_control_df = conn_control_df[['bodyId_pre','x_post','y_post','z_post','bodyId_post']]\
    .rename(columns={'x_post': 'x', 'y_post': 'y', 'z_post': 'z'})

hex_control_df = find_rel_hex(hex_red_df, syn_control_df, neuropil)
hex_control_df['hex_names'] = 100*(hex_control_df['hex1_id']+hex0) + hex_control_df['hex2_id']+hex0

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/183861 [00:00<?, ?it/s]

In [7]:
hex_bid_control_df = hex_control_df.groupby(['bodyId_post','hex_names'])['bodyId_pre'].count().reset_index(name='count')
hex_bid_control_df['frac'] = hex_bid_control_df.groupby('bodyId_post')['count'].transform(lambda x: x/x.sum())
print('Mean number of inputs: %.1f'%(hex_bid_control_df.groupby('bodyId_post')['count'].sum().mean()))

hex_count_control_df = hex_bid_control_df.groupby('hex_names')['frac'].sum().reset_index(name='frac')
hex_count_control_df['frac'] = hex_count_control_df['frac']/n_out
hex_count_control_df['hex1_id'] = (hex_count_control_df['hex_names'].values/100).astype(int)-hex0
hex_count_control_df['hex2_id'] = hex_count_control_df['hex_names']-100*(hex_count_control_df['hex1_id']+hex0)-hex0

Mean number of inputs: 216.6
Control variance: 0.0298


In [8]:
hex_count_control_df['frac'].sum()

Hex area: 36, small


In [11]:
hex_area = (hex_count_control_df['hex1_id'].max()-hex_count_control_df['hex1_id'].min())*(hex_count_control_df['hex2_id'].max()-hex_count_control_df['hex2_id'].min())
print('Hex area: %d'%hex_area)

if hex_area>300:
    hex_size = 'large'
elif (hex_area<=300)&(hex_area>50):
    hex_size = 'medium'
else:
    hex_size = 'small'

Max fraction: 0.2288
Min fraction: 0.0229


In [ ]:
hex_size = 'small'

In [ ]:
cmax=np.max([0.1,hex_count_control_df['frac'].max()])
cmax

In [ ]:
fig = plot_col_heatmap(hex_count_control_df, 'frac', hex_size, cmax=cmax)
# fig.update_layout(title=dict(text=f'Mean of all connections into {instance_name[:-2]}', x=0.5, y=0.9))
fig.update_layout(title=dict(text=f'Mean of all connections into {instance_name[:-2]}', x=0.5, y=0.9))

fig.show()
fig.write_image(data_path / f'{instance_name}_{neuropil[:-3]}_from_all_2d.pdf')
fig.write_image(data_path / f'{instance_name}_{neuropil[:-3]}_from_all_2d.png')

### distribution of input connections from one cell type

In [12]:
input_instance = 'Mi4'+'_R'

Control variance explained by fit: 0.0007


In [13]:
hex_instance_df = fetch_n_find_rel_hex(hex_red_df, input_instance, instance_name, neuropil, batch_size=10)
hex_instance_df['hex_names'] = 100*(hex_instance_df['hex1_id']+hex0) + hex_instance_df['hex2_id']+hex0

In [16]:
hex_bid_instance_df = hex_instance_df.groupby(['bodyId_post','hex_names'])['bodyId_pre'].count().reset_index(name='count')
hex_bid_instance_df['frac'] = hex_bid_instance_df.groupby('bodyId_post')['count'].transform(lambda x: x/x.sum())
print('Mean number of inputs: %.1f'%(hex_bid_instance_df.groupby('bodyId_post')['count'].sum().mean()))

hex_count_instance_df = hex_bid_instance_df.groupby('hex_names')['frac'].sum().reset_index(name='frac')
hex_count_instance_df['frac'] = hex_count_instance_df['frac']/n_out
hex_count_instance_df['hex1_id'] = (hex_count_instance_df['hex_names'].values/100).astype(int)-hex0
hex_count_instance_df['hex2_id'] = hex_count_instance_df['hex_names']-100*(hex_count_instance_df['hex1_id']+hex0)-hex0

c:\Users\hoellerj\AppData\Local\anaconda3\envs\ol-connectome\Lib\site-packages\neuprint\client.py:609: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

c:\Users\hoellerj\AppData\Local\anaconda3\envs\ol-connectome\Lib\site-packages\neuprint\client.py:619: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



Mean number of inputs: 18.7


In [17]:
fig = plot_col_heatmap(hex_count_instance_df, 'frac', hex_size, cmax=cmax)
fig.update_layout(title=dict(text=f'Mean of {input_instance[:-2]} connections into {instance_name[:-2]}', x=0.5, y=0.9))

fig.show()
fig.write_image(data_path / f'{instance_name}_{neuropil[:-3]}_from_{input_instance[:-2]}_2d.pdf')
fig.write_image(data_path / f'{instance_name}_{neuropil[:-3]}_from_{input_instance[:-2]}_2d.png')

### radial and angular distributions

In [ ]:
# rad_dist_control_df, angle_dist_control_df = find_1d_input_distrs(hex_control_df)
rad_dist_df, angle_dist_df = find_1d_input_distrs(hex_instance_df)

In [ ]:
fig = plot_1d_distr(rad_dist_df, 'radius', 'frac')

fig.show()
fig.write_image(data_path / f'{instance_name}_{neuropil[:-3]}_from_{input_instance[:-2]}_rad.pdf')
fig.write_image(data_path / f'{instance_name}_{neuropil[:-3]}_from_{input_instance[:-2]}_rad.png')

In [ ]:
fig2 = plot_1d_polar(angle_dist_df, 'angle', 'frac')

fig2.show()
fig.write_image(data_path / f'{instance_name}_{neuropil[:-3]}_from_{input_instance[:-2]}_ang.pdf')
fig.write_image(data_path / f'{instance_name}_{neuropil[:-3]}_from_{input_instance[:-2]}_ang.png')